###  Random Forest Model B Training, & Hyper-parameter Tuning

**Organization:** Break Through Tech AI Studio / MediMate Foundation  
**Milestone:** October Model Development   
**Authors / Contributors:** Healthcare 1C Team  

---

### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import log_loss
from sklearn.metrics import accuracy_score
import pickle
from pathlib import Path

In [2]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "processed_output" / "X_processed.csv").exists())
DATA_DIR = ROOT / "processed_output"
MODEL_DIR = ROOT / "models"

### Safety Checks on Datasets

In [3]:
X_processed = pd.read_csv(DATA_DIR / "X_processed.csv", index_col=0)
y_frame = pd.read_csv(DATA_DIR / "y_processed.csv", index_col=0)
if y_frame.shape[1] != 1 or not X_processed.index.equals(y_frame.index):
    raise ValueError("Expected one target column aligned with the feature row index.")
if not X_processed.index.is_unique or not X_processed.columns.is_unique:
    raise ValueError("Feature row indices and column names must be unique.")
y_processed = y_frame.iloc[:, 0]
if set(y_processed.unique()) != {0, 1}:
    raise ValueError("Expected a binary 0/1 target without missing values.")
if not all(pd.api.types.is_numeric_dtype(t) for t in X_processed.dtypes):
    raise ValueError("Expected numeric processed features.")
if not np.isfinite(X_processed.to_numpy()).all():
    raise ValueError("Features must not contain missing or infinite values.")

In [4]:
# Baseline B retains all processed features, including demographics.
excluded_columns = []
print(f"Baseline B uses all {X_processed.shape[1]} features.")

Baseline B uses all 51 features.


### Dataset Statistics and Splitting

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed
)
print(f"Training: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print("Training class counts:", y_train.value_counts().to_dict())

Training: 351909 samples, 51 features
Test: 87978 samples, 51 features
Training class counts: {0: 337839, 1: 14070}


### Random Forest Modelling Structure

In [6]:
def train_test_RFC(X_train, X_test, y_train, y_test, estimators, min_sample_split, max_features, bootstrap_bool, weight, min_sample_leaf = 20, maxdepth = 10, njobs = -1):
    """
        Train and evaluate a Random Forest classifier.
    
        Args:
            X_train: Training feature data.
            X_test: Test feature data.
            y_train: Target values corresponding to X_train.
            y_test: Target values corresponding to X_test.
            estimators (int): Number of trees in the forest.
            min_sample_split (int): Minimum number of samples required to split an internal node.
            max_features (int, float, str, or None): Number or proportion of features considered at each split.
            bootstrap_bool (bool): Whether bootstrap sampling is used when constructing trees.
            weight: Class weighting strategy used to handle class imbalance.
            min_sample_leaf (int, optional): Minimum number of samples required in a leaf node. Defaults to 20.
            maxdepth (int, optional): Maximum depth of each decision tree. Defaults to 10.
            njobs (int, optional): Number of CPU cores used for parallel processing. -1 uses all available cores.
    
        Returns:
            tuple: The trained Random Forest model and the model's predictions for X_test.
    
        Raises:
            ValueError: If any input dataset is None or if the number of training samples differs
                from the number of training target values.
        """

    # Validation of input parameters
    if X_train is None or X_test is None or y_train is None or y_test is None:
        raise ValueError("Training and test data cannot be None.")

    if len(X_train) != len(y_train):
        raise ValueError("X_train and y_train must contain the same number of samples.")

    if len(X_test) != len(y_test):
        raise ValueError("X_test and y_test must contain the same number of samples.")

    if estimators <= 0:
        raise ValueError("estimators must be greater than zero.")

    if min_sample_split < 2:
        raise ValueError("min_sample_split must be at least 2.")

    if min_sample_leaf < 1:
        raise ValueError("min_sample_leaf must be at least 1.")

    if maxdepth is not None and maxdepth < 1:
        raise ValueError("maxdepth must be None or greater than zero.")

    if njobs == 0:
        raise ValueError("njobs cannot be zero.")
    
    # Create a random forest model
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    model = make_pipeline(RandomForestClassifier(
    n_estimators = estimators,       # Number of trees in the forest
    max_depth= maxdepth,           # Restrict depth to prevent overfitting
    min_samples_split = min_sample_split,
    min_samples_leaf = min_sample_leaf, # Smoothing the leaves helps reduce variance in noisy data.
    max_features = max_features,
    bootstrap= bootstrap_bool,            # Use bootstrapping to sample data for each tree
    random_state= 42,        # Ensures reproducible results across production runs
    n_jobs= njobs,             # Use all available CPU cores for parallel processing
    class_weight= weight
))

    # Fit the model to the training data
    model.fit(X_train, y_train)

    #Predict the probability of the class of examples of the test data

    proba_predict = model.predict_proba(X_test)

    #Computing the log loss of the probability (proba_predict) by comparing 
    #the actual labels of the test data to the probability predictions

    logloss = log_loss(y_test, proba_predict)

    #Making predictions on the test data, to evaluate the accuracy score
    
    class_label_predict = model.predict(X_test)

    #Computing the accuracy score of the model by comparing the y_test
    #with the class label predictions generated from the step above
    
    accscore = accuracy_score(y_test, class_label_predict)
    
    return {"model": model, "predicted_proba": proba_predict, 'accuracy_score': accscore, 'log_loss': logloss, "predicted_classes": class_label_predict}

### Model Pickling

In [7]:
def pickling(model, model_name):
    """
    Save a model to a pickle file, then load and return it.

    Args:
        model: The object to serialize. It must be pickle-compatible.
        model_name (str or os.PathLike): Path to the pickle file.

    Returns:
        The model deserialized from the file.

    Raises:
        RuntimeError: If saving or loading the model fails.

    Note:
        Only load pickle files from trusted sources; unpickling untrusted
        files can execute arbitrary code.
    """
    try:
        with open(model_name, "wb") as handle:
            pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)
    except (OSError, pickle.PicklingError, AttributeError, TypeError, RecursionError) as exc:
        raise RuntimeError(f"Failed to save model to {model_name!r}.") from exc

    # Only load pickle files produced here or from another trusted source.
    try:
        with open(model_name, "rb") as handle:
            return pickle.load(handle)
    except (OSError, pickle.UnpicklingError, EOFError, AttributeError, ImportError, IndexError) as exc:
        raise RuntimeError(f"Failed to load model from {model_name!r}.") from exc

###  Hyperparameter Optimization

This section tunes a Random Forest using all processed features, including demographics. It selects parameters by stratified cross-validated average precision on the training data, then evaluates the selected model once on the held-out test split. Run the following cell to save the fitted model and its results.

In [8]:

from pathlib import Path
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score, balanced_accuracy_score
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, log_loss
import hashlib
import json
import platform
import sklearn

# Configuration
ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "processed_output" / "X_processed.csv").exists()
)

DATA_DIR = ROOT / "processed_output"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_ITER = 8
CV_FOLDS = 3
N_JOBS = 2

# Load processed datasets
x_path = DATA_DIR / "X_processed.csv"
y_path = DATA_DIR / "y_processed.csv"
source_metadata_path = DATA_DIR / "dataset_metadata.json"

for path in (x_path, y_path, source_metadata_path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required processed dataset file not found: {path}"
        )

X_all = pd.read_csv(x_path, index_col=0)
y_frame_all = pd.read_csv(y_path, index_col=0)

# Validate datasets
if y_frame_all.shape[1] != 1 or not X_all.index.equals(y_frame_all.index):
    raise ValueError("Expected one target column aligned with feature rows.")

if not X_all.index.is_unique or not X_all.columns.is_unique:
    raise ValueError("Feature row indices and column names must be unique.")

y_all = y_frame_all.iloc[:, 0]

if y_all.isna().any() or set(y_all.unique()) != {0, 1}:
    raise ValueError("Expected a complete binary 0/1 target.")

if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in X_all.dtypes):
    raise ValueError("Expected numeric processed features.")

if not np.isfinite(X_all.to_numpy()).all():
    raise ValueError("Features must not contain missing or infinite values.")

# Baseline B retains all processed features, including demographics.
excluded_columns = []

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_all
)

# Hyperparameter search space
param_distributions = {
    "n_estimators": [100, 200],
    "max_depth": [8, 12, 16, None],
    "min_samples_split": [2, 10, 50],
    "min_samples_leaf": [5, 20, 50, 100],
    "max_features": ["sqrt", 0.5],
    "class_weight": [None, "balanced_subsample"],
    "max_samples": [0.5, 0.75, None],
}

# Randomized hyperparameter optimization
search = RandomizedSearchCV(
    RandomForestClassifier(
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=1
    ),
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring="average_precision",
    cv=StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE
    ),
    refit=False,
    n_jobs=N_JOBS,
    pre_dispatch=N_JOBS,
    random_state=RANDOM_STATE,
    error_score="raise",
    verbose=2
)

print("Starting Random Forest hyperparameter optimization...")
search.fit(X_train, y_train)

# Save cross-validation results
pd.DataFrame(search.cv_results_).to_csv(
    MODEL_DIR / "random_forest_baseline_b_cv_results.csv",
    index=False
)

print("Best CV average precision:", search.best_score_)
print("Best parameters:", search.best_params_)

# Train optimized model
best_model = RandomForestClassifier(
    **search.best_params_,
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS
).fit(X_train, y_train)

# Generate predictions
y_probability = best_model.predict_proba(X_test)[:, 1]
y_prediction = (y_probability >= 0.5).astype(np.int8)

# Evaluate model
test_metrics = {
    "average_precision": float(
        average_precision_score(y_test, y_probability)
    ),
    "roc_auc": float(
        roc_auc_score(y_test, y_probability)
    ),
    "accuracy": float(
        accuracy_score(y_test, y_prediction)
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(y_test, y_prediction)
    ),
    "adverse_recall_at_0_5": float(
        recall_score(y_test, y_prediction, zero_division=0)
    ),
    "log_loss": float(
        log_loss(y_test, y_probability, labels=[0, 1])
    ),
}

test_confusion = confusion_matrix(
    y_test,
    y_prediction,
    labels=[0, 1]
).tolist()

# Save trained model
model_path = MODEL_DIR / "random_forest_baseline_b_best.pkl"

with model_path.open("wb") as handle:
    pickle.dump(
        best_model,
        handle,
        protocol=pickle.HIGHEST_PROTOCOL
    )

# Reload saved model
with model_path.open("rb") as handle:
    reloaded_model = pickle.load(handle)

# Verify saved model predictions
reloaded_probability = reloaded_model.predict_proba(X_test)[:, 1]

max_difference = np.max(
    np.abs(y_probability - reloaded_probability)
)

print("Maximum prediction difference:", max_difference)
print(
    "Exact prediction match:",
    np.array_equal(y_probability, reloaded_probability)
)

# Allow negligible floating-point differences
np.testing.assert_allclose(
    reloaded_probability,
    y_probability,
    rtol=1e-12,
    atol=1e-12
)

print("Saved model verification passed!")

# Generate source file hashes
source_hashes = {}

for path in (x_path, y_path, source_metadata_path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    source_hashes[path.name] = digest.hexdigest()

# Save model metadata
metadata = {
    "baseline": "B",
    "target": "adverse_365",
    "positive_class": 1,
    "source_sha256": source_hashes,
    "feature_columns": X_all.columns.tolist(),
    "excluded_columns": excluded_columns,
    "selection_metric": "average_precision",
    "best_parameters": search.best_params_,
    "best_cv_average_precision": float(search.best_score_),
    "candidate_count": N_ITER,
    "cv_folds": CV_FOLDS,
    "test_metrics": test_metrics,
    "test_confusion_matrix": test_confusion,
    "decision_threshold": 0.5,
    "test_size": 0.2,
    "stratified_split": True,
    "training_rows": int(len(y_train)),
    "test_rows": int(len(y_test)),
    "random_state": RANDOM_STATE,
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
}

metadata_path = MODEL_DIR / "random_forest_baseline_b_metadata.json"

metadata_path.write_text(
    json.dumps(metadata, indent=2) + "\n",
    encoding="utf-8"
)

# Final results
print("\n========== OPTIMIZATION RESULTS ==========")
print(f"Best CV average precision: {search.best_score_:.6f}")
print("Best parameters:", search.best_params_)
print("Test metrics:", test_metrics)
print("Test confusion matrix [true rows, predicted columns]:", test_confusion)
print("Saved model, CV results, and metadata under models/.")

Starting Random Forest hyperparameter optimization...
Fitting 3 folds for each of 8 candidates, totalling 24 fits


KeyboardInterrupt: 

## Summary



This notebook tunes a secondary baseline Random Forest classifier using all 51 processed features, including demographics, to allow for performance comparisons. Key steps include:

- **Data ingestion and validation:** Loads the processed features and binary target, checks row alignment, unique indices and columns, numeric feature types, and finite values.
- **Feature selection:** Retains every feature, including demographic features, for Baseline B.
- **Data splitting:** Uses a stratified 80/20 train/test split with random state 42.
- **Hyperparameter optimization:** Runs an 8-candidate randomized search with 3-fold stratified cross-validation, selecting by average precision on the training data only. The best parameters are `n_estimators=100`, `min_samples_split=50`, `min_samples_leaf=20`, `max_samples=0.75`, `max_features='sqrt'`, `max_depth=None`, and `class_weight=None`. Best CV average precision is **0.100801**.
- **Held-out evaluation:** Test average precision is **0.107480**, ROC AUC is **0.700067**, and log loss is **0.157088**. At the default 0.5 threshold, adverse recall is **0.0**; the confusion matrix is `[[84461, 0], [3517, 0]]`. Ranking performance does not imply positive-case detection at this threshold.
- **Model persistence:** Saves the fitted forest, all CV results, and evaluation metadata to `models/random_forest_baseline_b_best.pkl`, `models/random_forest_baseline_b_cv_results.csv`, and `models/random_forest_baseline_b_metadata.json`. The saved estimator's expected feature order is recorded in its metadata.